# Lopez-Salido, Stein & Zakrajsek (2017): Extension Walkthrough
### Rebuilding the replication on the Aaa-Treasury spread

The paper builds its credit-market sentiment proxy from the **Baa**-Treasury
spread — Moody's *lowest* investment-grade yield against the 10-year Treasury.
That choice is deliberate: Baa sits at the boundary between investment grade
and junk, so it is where reach-for-yield behaviour should show up most.

**Our extension asks what happens if we use the safest end of the credit
curve instead.** We rebuild every exhibit — Figures I and II, Tables I and II —
substituting the **Aaa**-Treasury spread for Baa throughout, and compare. The
pipeline supports this natively: every builder takes a `spread_col` argument,
and `main()` in each replication script emits an `_aaa` variant alongside the
original.

| Exhibit | Baa (replication) | Aaa (extension) |
|---|---|---|
| Figure I | `figure_1_replication.pdf` | `figure_1_aaa_replication.pdf` |
| Table I | `table_1_replication.tex` | `table_1_aaa_replication.tex` |
| Table II | `table_2_replication.tex` | `table_2_aaa_replication.tex` |
| Figure II | `figure_2_replication.pdf` | `figure_2_aaa_replication.pdf` |

**What to expect.** If LSZ's mechanism is really about *credit-risk* sentiment
rather than the general level of interest rates, the Aaa version should be a
weaker version of the same story: the same signs, but smaller and less
significant, because Aaa issuers are barely exposed to the default-risk
repricing that drives the Baa spread.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
import replicate_figure_1 as f1
import replicate_figure_2 as f2
import replicate_table_1 as t1
import replicate_table_2 as t2
from settings import config

PROCESSED_DATA_DIR = Path(config("PROCESSED_DATA_DIR"))
OUTPUT_DIR = Path(config("OUTPUT_DIR"))

BAA, AAA = "BAA_Treasury_spread", "AAA_Treasury_spread"
print(f"Replication window: {t1.REP_START}-{t1.REP_END}   extension end: {t1.EXT_END}")

## 1. Figure I — the two spreads side by side

Figure I plots the credit spread with NBER recessions shaded. Rebuilt on Aaa,
the countercyclical shape survives but the amplitude shrinks: Aaa spikes in
recessions too, just far less violently than Baa, since a Aaa issuer's default
risk barely moves even in severe downturns. The gap between the two lines is
the compensation investors demand for bearing *credit* risk specifically —
which is exactly the quantity LSZ's sentiment story is about.

In [ ]:
monthly = pd.read_parquet(PROCESSED_DATA_DIR / "fred_final_series_monthly.parquet")

fig, spread_aaa = f1.plot_figure_1(monthly, spread_col=AAA)
plt.show()

window = monthly.loc[str(t1.REP_START):str(t1.REP_END)]
print("Spread summary over the replication window (percentage points):")
print(window[[BAA, AAA]].describe().loc[["mean", "std", "min", "max"]].round(2))

## 2. Table I — does the credit-beats-equity result survive?

Table I forecasts next-year real GDP-per-capita growth with the change in the
credit spread and the S&P 500 total return. Column (3) is the paper's headline:
with both in, credit stays significant and equity collapses.

We re-run all three columns on the Aaa spread and print the key coefficients
next to the Baa ones.

In [ ]:
base = ["gdp_pc_growth"]
specs = {
    "(1) credit only": ["d_credit_spread"] + base,
    "(2) equity only": ["sp_return"] + base,
    "(3) both + controls": ["d_credit_spread", "sp_return", "d_treasury_3mo",
                            "d_treasury_10yr", "CPI_inflation"] + base,
}

rows = []
for spread_label, spread_col in [("Baa", BAA), ("Aaa", AAA)]:
    df = t1.build_panel(spread_col=spread_col)
    for col_label, regressors in specs.items():
        res = t1.run_regression(df, regressors, t1.REP_START, t1.REP_END)
        for var in ("d_credit_spread", "sp_return"):
            if var in res.params.index:
                rows.append({
                    "spread": spread_label,
                    "column": col_label,
                    "regressor": var,
                    "coef": round(res.params[var], 3),
                    "p": round(res.pvalues[var], 3),
                })

table_1_compare = pd.DataFrame(rows)
print(table_1_compare.to_string(index=False))

Read the `d_credit_spread` rows: the Aaa coefficient should keep the negative
sign (a widening spread still forecasts weaker growth) but come out
attenuated relative to Baa. That attenuation is the point of the extension —
the forecasting power is concentrated in the risky end of the credit curve, not
in duration or the general level of yields.

## 3. Table II — the sentiment two-step on Aaa

Table II is the analytical core. The first stage forecasts the change in the
spread from twice-lagged sentiment (the high-yield issuance share and the
spread level); the second stage regresses growth on the *fitted* values.

We estimate the whole system on Aaa and compare the first-stage coefficients
and the headline second-stage term against Baa.

In [ ]:
summary_rows = []
for spread_label, spread_col in [("Baa", BAA), ("Aaa", AAA)]:
    df = t2.build_panel(spread_col=spread_col)
    res = t2.run_table_2(df, t2.REP_START, t2.REP_END)
    aux, col3 = res["aux_spread"], res["col3"]
    summary_rows.append({
        "spread": spread_label,
        "a1 lnHYS_{t-2}": round(aux.params["ln_hys_lag2"], 3),
        "a2 s_{t-2}": round(aux.params["spread_lag2"], 3),
        "aux R2": round(aux.rsquared, 3),
        "col3 d_s_hat": round(col3.params["d_s_hat"], 3),
        "col3 p": round(col3.pvalues["d_s_hat"], 3),
        "col3 R2": round(col3.rsquared, 3),
    })

table_2_compare = pd.DataFrame(summary_rows).set_index("spread")
print(table_2_compare.to_string())

Three things to check in that table:

1. **`a1` stays positive** — a high past high-yield issuance share still
   forecasts a widening spread, even an Aaa one. Froth in the junk market
   spills over into the pricing of safe credit.
2. **`a2` stays negative** — spreads mean-revert at both ends of the curve.
3. **`col3 d_s_hat` stays negative** — fitted spread widening still forecasts
   lower growth, but the Aaa version is the weaker of the two, and the
   `aux R2` shows sentiment explains less of the Aaa spread's variation.

The generated-regressor caveat from `replicate_table_2` applies equally here:
the second-step standard errors do not propagate first-stage sampling
uncertainty, so the p-values are optimistic for both spreads.

## 4. Figure II — sentiment vs. growth, Aaa version

Figure II scatters credit-market sentiment at $t-2$ against realized growth at
$t$, both orthogonalized against column (1)'s other regressor, with the fitted
line from Table II column (1) and influential observations starred.

In [ ]:
df_aaa = t2.build_panel(spread_col=AAA)
fig = f2.plot_figure_2(df_aaa, t2.REP_START, t2.REP_END)
plt.show()

res1_aaa = t2.run_table_2(df_aaa, t2.REP_START, t2.REP_END)["col1"]
influential = f2.find_influential(res1_aaa, "d_s_hat")
print(f"Slope on fitted sentiment (Aaa): {res1_aaa.params['d_s_hat']:.3f}")
print(f"Influential years (|DFBETAS| > 2/sqrt(T)): {sorted(int(y) for y in influential)}")

The starred years are the observations whose removal would most move the
slope. Comparing this list against the Baa version tells you whether the
Aaa relationship rests on the same historical episodes (Depression and
post-war years, typically) or on a different, thinner set — which matters for
how much weight the Aaa result can carry.

## 5. Generated output files

`doit` emits both variants of every exhibit. The Aaa files carry an `aaa`
tag; the Baa ones keep the original unsuffixed names.

In [ ]:
produced = sorted(p.name for p in OUTPUT_DIR.glob("*aaa*"))
print(f"{len(produced)} Aaa extension files in {OUTPUT_DIR.name}/:")
for name in produced:
    print("  ", name)

## Summary

Rebuilding the replication on the Aaa-Treasury spread reproduces the *shape* of
every LSZ result — countercyclical spreads, credit beating equities as a growth
predictor, froth forecasting widening, and fitted sentiment forecasting slower
growth — while attenuating the magnitudes. That is the informative outcome: the
mechanism is not an artefact of duration or the level of interest rates, since
it survives at the safe end of the curve, but its strength scales with exposure
to credit risk, which is why the paper's choice of Baa is the right one for
measuring sentiment. The Aaa variant is best read as a placebo-flavoured
robustness check that the Baa results pass.